# Verificação do bundle publicado no Drive

Prova, ponta a ponta, que o `bundle.tar.gz` que está no Google Drive é o mesmo
que saiu daqui e que o Colab consegue treinar a partir dele.

O teste não olha o arquivo local: ele **baixa de volta** do Drive pelo rclone e
valida o que voltou. Um bundle truncado no upload, um `--drive-chunk-size` mal
ajustado ou uma pasta homônima passariam despercebidos numa checagem local.

Ordem: baixa → confere o digest do manifesto contra o sidecar → abre o dataset →
confere o tier de features, o período e os splits.

In [ ]:
import json
import shutil
import subprocess
import tarfile
import tempfile
from pathlib import Path

import pandas as pd

from allsky.bundle import validate_bundle
from allsky.features import resolve_feature_set

RCLONE = shutil.which("rclone") or str(Path.home() / ".local" / "bin" / "rclone")
REMOTE_BUNDLE = "labmim:labmim/allsky-mm/bundle.tar.gz"
FEATURE_SET = "bare"


def rclone(*args: str) -> str:
    """Run rclone and return its stdout, raising on a non-zero exit."""
    completed = subprocess.run([RCLONE, *args], capture_output=True, text=True, check=True)
    return completed.stdout

## 1. O que está publicado

`lsjson` traz tamanho e digest do lado do Drive, antes de qualquer download.

In [ ]:
remoto = json.loads(rclone("lsjson", "--hash", "labmim:labmim/allsky-mm"))
for item in remoto:
    print(f"{item['Name']:<24} {item['Size']:>14,} B  {item.get('ModTime', '')}")
    for algo, valor in (item.get("Hashes") or {}).items():
        print(f"    {algo}: {valor}")

## 2. Traz de volta

Para um diretório temporário, nunca por cima do artefato local — senão o teste
passaria a comparar o arquivo consigo mesmo.

In [ ]:
temporario = tempfile.mkdtemp(prefix="verifica-bundle-")
destino = Path(temporario)

rclone("copy", "--progress", REMOTE_BUNDLE, str(destino))
baixado = destino / "bundle.tar.gz"

print(f"baixado: {baixado}")
print(f"tamanho: {baixado.stat().st_size:,} bytes")

## 3. O digest do manifesto

`validate_bundle` recalcula o sha256 do conteúdo do `manifest.parquet` e compara
com o `manifest_sha256` que o sidecar gravou na exportação. Se o upload corrompeu
um byte, é aqui que aparece.

In [ ]:
relatorio = validate_bundle(baixado)
print(json.dumps(relatorio, indent=2, ensure_ascii=False, default=str)[:2000])

## 4. O dataset que o Colab vai ler

In [ ]:
extraido = destino / "extraido"
with tarfile.open(baixado) as tar:
    tar.extractall(extraido, filter="data")

manifesto_path = next(extraido.rglob("manifest.parquet"))
raiz = manifesto_path.parent
manifesto = pd.read_parquet(manifesto_path)

print(f"raiz do bundle: {raiz.relative_to(extraido)}")
print(f"linhas: {len(manifesto):,}   colunas: {len(manifesto.columns)}")
print(f"período: {manifesto['timestamp_utc'].min()} .. {manifesto['timestamp_utc'].max()}")
print(f"dias distintos: {manifesto['day_id'].nunique()}")

## 5. O tier de features

O motivo de o tier existir: a MetSENS1 falhou por completo em 2026-08-10 13:05 e
levou o barômetro junto, então `pressure_mbar` não pode estar aqui — se estivesse,
o filtro de linhas finitas teria descartado todo o overlap com os dias novos de
câmera.

In [ ]:
esperadas = resolve_feature_set(FEATURE_SET)
presentes = [c for c in esperadas if c in manifesto.columns]

print(f"tier {FEATURE_SET!r}: {len(esperadas)} features")
for nome in esperadas:
    marca = "ok" if nome in manifesto.columns else "AUSENTE"
    print(f"  [{marca}] {nome}")

assert presentes == esperadas, f"faltam no manifesto: {set(esperadas) - set(presentes)}"
assert "pressure_mbar" not in manifesto.columns, "o barômetro morto voltou ao manifesto"

nao_finitas = int((~manifesto[esperadas].map(pd.api.types.is_number).all(axis=1)).sum())
print(f"\nlinhas com feature não numérica: {nao_finitas}")
assert not manifesto[esperadas].isna().to_numpy().any(), "NaN sobreviveu ao filtro"

## 6. Os splits

Split cronológico por dia com gap: nenhum dia pode aparecer em dois conjuntos, e
todo dia de treino tem que preceder todo dia de teste. Embaralhar aqui seria
vazamento — o frame de t e o de t+60 s são quase o mesmo dado.

In [ ]:
splits = json.loads((raiz / "splits.json").read_text(encoding="utf-8"))
assignment = splits["assignment"]
dias = {}
for dia, nome in assignment.items():
    dias.setdefault(nome, set()).add(dia)
dias = {nome: dias.get(nome, set()) for nome in ("train", "val", "test")}

for nome, valores in dias.items():
    if valores:
        print(f"{nome:<6} {len(valores):>3} dias   {min(valores)} .. {max(valores)}")

for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (dias[a] & dias[b]), f"{a} e {b} compartilham dias"

if dias["train"] and dias["test"]:
    assert max(dias["train"]) < min(dias["test"]), "treino invade o futuro do teste"

print(f"\nsplit_id: {splits.get('split_id')}")

## 7. Os embeddings

In [ ]:
emb_meta = next(extraido.rglob("embeddings/embeddings.meta.json"), None)

if emb_meta is None:
    print("bundle sem embeddings — o Colab teria que recalcular")
else:
    meta = json.loads(emb_meta.read_text(encoding="utf-8"))
    shards = sorted(emb_meta.parent.glob("*.safetensors"))
    print(json.dumps(meta, indent=2, ensure_ascii=False)[:900])
    print(f"\nshards: {len(shards)}")
    print(f"bytes:  {sum(s.stat().st_size for s in shards):,}")

## 8. Limpeza

O diretório temporário some; o Drive fica intocado (nada aqui escreve nele).

In [ ]:
import shutil

shutil.rmtree(temporario, ignore_errors=True)
print(f"removido: {temporario}")
print("\nbundle no Drive verificado ponta a ponta.")